# Experiment 4: Neuroabundance Scheduling

To determine whether initial over- or under-abundance of neurons results in any meaningful accuracy difference. Neural overabundance is often beneficial in mammals, so it is interesting to test this. The scheduling begins at one fixed width for 15 epochs, then 75 epochs linearly scheduled adaption, then 10 epochs at the final width. CIFAR-10 using two-hidden-layer networks of the form `[input_dim, n, n, output_dim]`, and intrinsic length and psi-vector are trainable (latter has decay).

Important that the hidden-width adjustment order is always **h2 first, then h1**. For `[input_dim, h1, h2, output_dim]`, this means layer `1` is adjusted before layer `0`, otherwise error from psi-vector can propagate.

In [ ]:
import os
import pickle as pkl
import platform
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from Dependencies import *



In [ ]:
# -------------------------
# Experiment 4 configuration
# -------------------------

WIDTHS = [7, 10, 25, 50, 75, 100] ## Added 7 and 10 later to observe what neuro-under-abundance does
INPUT_DIM = 32 * 32 * 3
OUTPUT_DIM = 10

REPEATS = 4
WARMUP_EPOCHS = 15
ADAPTATION_EPOCHS = 75
SETTLING_EPOCHS = 10
TOTAL_EPOCHS = WARMUP_EPOCHS + ADAPTATION_EPOCHS + SETTLING_EPOCHS

LEARNING_RATE = 1e-3
BATCH_SIZE = 48
PSI_DECAY = 1e-1
ADAMW_WEIGHT_DECAY = 1e-3
WEIGHT_INIT = "orthogonal"

INTRINSIC_LENGTH_APPROACH = "TRAINABLE"
LINEAR_CORRECTION_APPROACH = "TRAINABLE+DECAY"
POSITIVE_INTRINSIC_LENGTH = True
INIT_INTRINSIC_LENGTH = 1e-6
TANH_EPSILON = 1e-3

NORMALISATION = True
COMPUTE_CIFAR_NORMALISATION_IF_MISSING = True


RANDOM_SEED_BASE = 170000
DEVICE = try_gpu(output=True, i=0)

SAVE_DIR = Path("./Saved_Models/Experiment 4/Neuroabundance/")
RUN_SAVE_DIR = SAVE_DIR / "per_run_histories"
FIGURE_DIR = SAVE_DIR / "figures"
FORCE_RERUN = False
CACHE_COMPLETED_RUNS = True

SAVE_DIR.mkdir(parents=True, exist_ok=True)
RUN_SAVE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Experiment 4 settings")
print("---------------------")
print(f"Dataset: CIFAR-10")
print(f"Widths: {WIDTHS}")
print(f"Repeats per setup: {REPEATS}")
print(f"Epochs: warmup={WARMUP_EPOCHS}, adaptation={ADAPTATION_EPOCHS}, settling={SETTLING_EPOCHS}, total={TOTAL_EPOCHS}")
print(f"Device: {DEVICE}")
print(f"Isotropic model: intrinsic={INTRINSIC_LENGTH_APPROACH}, linear correction={LINEAR_CORRECTION_APPROACH}")
print("Width adjustment rule: h2 first, then h1")


In [ ]:
# -------------------------
# Runtime/GPU facts
# -------------------------

def print_runtime_facts(device):
    print("Runtime facts")
    print("-------------")
    print(f"Python: {platform.python_version()}")
    print(f"PyTorch: {torch.__version__}")
    print(f"Platform: {platform.platform()}")
    print(f"Selected device: {device}")

    if torch.cuda.is_available() and str(device).startswith("cuda"):
        idx = device.index if device.index is not None else 0
        props = torch.cuda.get_device_properties(idx)
        print("CUDA available: True")
        print(f"CUDA version used by PyTorch: {torch.version.cuda}")
        print(f"GPU name: {torch.cuda.get_device_name(idx)}")
        print(f"GPU total memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"GPU allocated memory now: {torch.cuda.memory_allocated(idx) / 1024**3:.3f} GB")
        print(f"GPU reserved memory now: {torch.cuda.memory_reserved(idx) / 1024**3:.3f} GB")
    else:
        print("CUDA available: False")

print_runtime_facts(DEVICE)


In [ ]:
# -------------------------
# CIFAR-10 data
# -------------------------

def compute_cifar_normalisation(root="./data"):
    raw_transform = transforms.Compose([transforms.ToTensor()])
    raw_train = datasets.CIFAR10(root=root, train=True, download=True, transform=raw_transform)
    loader = DataLoader(raw_train, batch_size=512, shuffle=False, num_workers=0)

    pixel_sum = None
    pixel_sq_sum = None
    n = 0

    for xb, _ in loader:
        if pixel_sum is None:
            pixel_sum = torch.zeros_like(xb[0])
            pixel_sq_sum = torch.zeros_like(xb[0])
        pixel_sum += xb.sum(dim=0)
        pixel_sq_sum += (xb ** 2).sum(dim=0)
        n += xb.shape[0]

    mean = pixel_sum / n
    var = torch.clamp(pixel_sq_sum / n - mean ** 2, min=1e-12)
    inv_std = 1.0 / torch.sqrt(var)
    return {"mean": mean.to(torch.float32), "inverse stddev": inv_std.to(torch.float32)}

class PerPixelNormalize:
    def __init__(self, path="./CIFAR_normalisations.pkl"):
        path = Path(path)
        if path.exists():
            normaliser_dictionary = pkl.load(open(path, "rb"))
        elif COMPUTE_CIFAR_NORMALISATION_IF_MISSING:
            print("CIFAR_normalisations.pkl not found. Computing CIFAR-10 per-pixel normalisation from training data.")
            normaliser_dictionary = compute_cifar_normalisation(root="./data")
            with open(path, "wb") as f:
                pkl.dump(normaliser_dictionary, f)
            print(f"Saved computed normalisation to {path}")
        else:
            raise FileNotFoundError(f"Could not find {path}.")

        self.mean = normaliser_dictionary["mean"].to(torch.float32)
        self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

    def __call__(self, tensor):
        return (tensor - self.mean) * self.inv_std

if NORMALISATION:
    print("Using CIFAR-10 per-pixel normalisation")
    transform = transforms.Compose([transforms.ToTensor(), PerPixelNormalize()])
else:
    print("Not using CIFAR-10 normalisation")
    transform = transforms.Compose([transforms.ToTensor()])

cifar_train = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
cifar_test = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(cifar_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
test_loader = DataLoader(cifar_test, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f"Training examples: {len(cifar_train)}")
print(f"Test examples: {len(cifar_test)}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")


In [ ]:
# -------------------------
# Model definitions
# -------------------------

def build_isotropic_network(n_start):
    architecture = [INPUT_DIM, n_start, n_start, OUTPUT_DIM]
    network = IsotropicTanhMLP(
        layers=architecture,
        flatten=True,
        unflatten_shape=None,
        intrinsic_length_approach=INTRINSIC_LENGTH_APPROACH,
        linear_correction_approach=LINEAR_CORRECTION_APPROACH,
        positive_intrinsic_length=POSITIVE_INTRINSIC_LENGTH,
        init_intrinsic_length=INIT_INTRINSIC_LENGTH,
        tanh_epsilon=TANH_EPSILON,
        device=DEVICE,
        dtype=torch.get_default_dtype(),
    )
    network.simple_initialiser(weight_init=WEIGHT_INIT)
    return network

In [ ]:
# -------------------------
# Helper functions
# -------------------------

def set_repeat_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def count_total_parameters(model):
    return int(sum(parameter.numel() for parameter in model.parameters()))

def count_nonzero_parameters(model, threshold=1e-8):
    with torch.no_grad():
        return int(sum((parameter.detach().abs() > threshold).sum().item() for parameter in model.parameters()))

def width_schedule(n_start, n_target):
    # Inclusive endpoint schedule. Entry 0 is n_start; entry -1 is n_target.
    return np.rint(np.linspace(n_start, n_target, ADAPTATION_EPOCHS + 1)).astype(int)

def phase_for_epoch(epoch_idx):
    if epoch_idx < WARMUP_EPOCHS:
        return "warmup"
    if epoch_idx < WARMUP_EPOCHS + ADAPTATION_EPOCHS:
        return "adaptation"
    return "settling"

def target_width_for_epoch(epoch_idx, n_start, n_target):
    phase = phase_for_epoch(epoch_idx)
    if phase == "warmup":
        return int(n_start)
    if phase == "adaptation":
        schedule = width_schedule(n_start, n_target)
        adaptation_step = epoch_idx - WARMUP_EPOCHS + 1  # 1..ADAPTATION_EPOCHS
        return int(schedule[adaptation_step])
    return int(n_target)

def adjust_hidden_widths_to_target(network, optimiser, target_width, verbose=False):
    # Requested order: h2 first, then h1.
    operations = []

    # h2 first: architecture[2], adjusted by layer 1.
    while network.architecture[2] < target_width:
        network.neurogenerate(layer=1, verbose=verbose, optimiser=optimiser)
        operations.append("generate_h2")
    while network.architecture[2] > target_width:
        network.neurodegenerate(layer=1, verbose=verbose, optimiser=optimiser)
        operations.append("degenerate_h2")

    # h1 second: architecture[1], adjusted by layer 0.
    while network.architecture[1] < target_width:
        network.neurogenerate(layer=0, verbose=verbose, optimiser=optimiser)
        operations.append("generate_h1")
    while network.architecture[1] > target_width:
        network.neurodegenerate(layer=0, verbose=verbose, optimiser=optimiser)
        operations.append("degenerate_h1")

    return operations



def run_cache_path(setup_id, repeat_idx):
    safe_setup = setup_id.replace("→", "to").replace(" ", "_").replace("/", "-")
    return RUN_SAVE_DIR / f"{safe_setup}_repeat_{repeat_idx:02d}.pkl"


In [ ]:
# -------------------------
# Training/evaluation functions
# -------------------------

def add_psi_decay_if_needed(network, error, lambda_psi=PSI_DECAY):
    if getattr(network, "linear_correction_approach", "").upper() == "TRAINABLE+DECAY":
        psi_decay = 0.0
        for psi in network.psi_parameters:
            psi_decay = psi_decay + psi.square().sum()
        error = error + lambda_psi * psi_decay
    return error

def train_one_epoch(network, train_loader, optimiser, loss_fn, device, lambda_psi=PSI_DECAY):
    network.train()
    running_loss = []
    running_accuracy = []
    for batch_data, batch_labels in train_loader:
        batch_data = batch_data.to(device)
        batch_labels = batch_labels.to(device)
        optimiser.zero_grad(set_to_none=True)
        logits = network(batch_data)
        error = loss_fn(logits, batch_labels)
        error = add_psi_decay_if_needed(network, error, lambda_psi=lambda_psi)
        error.backward()
        optimiser.step()
        running_loss.append(float(error.detach().cpu().item()))
        running_accuracy.append(100.0 * (logits.argmax(dim=1) == batch_labels).float().mean().item())
    return float(np.mean(running_loss)), float(np.mean(running_accuracy))

@torch.no_grad()
def evaluate(network, data_loader, loss_fn, device):
    network.eval()
    running_loss = []
    correct = 0
    total = 0
    for batch_data, batch_labels in data_loader:
        batch_data = batch_data.to(device)
        batch_labels = batch_labels.to(device)
        logits = network(batch_data)
        loss = loss_fn(logits, batch_labels)
        running_loss.append(float(loss.detach().cpu().item()))
        correct += int((logits.argmax(dim=1) == batch_labels).sum().item())
        total += int(batch_labels.numel())
    return float(np.mean(running_loss)), 100.0 * correct / total


In [ ]:
# -------------------------
# Setup table
# -------------------------

setup_rows = []
for n_start in WIDTHS:
    for n_target in WIDTHS:
        setup_rows.append({
            "setup_id": f"isotropic_{n_start}to{n_target}",
            "family": "isotropic_neuroabundance",
            "n_start": n_start,
            "n_target": n_target,
            "label": f"iso {n_start}→{n_target}",
        })

setups_df = pd.DataFrame(setup_rows)
display(setups_df)
print(f"Total setups: {len(setups_df)}")
print(f"Total planned runs: {len(setups_df) * REPEATS}")


In [ ]:
# -------------------------
# Main experiment
# -------------------------

loss_fn = nn.CrossEntropyLoss()
all_epoch_records = []
all_run_records = []

for setup_index, setup in setups_df.iterrows():
    setup_id = setup["setup_id"]
    family = setup["family"]
    n_start = int(setup["n_start"])
    n_target = int(setup["n_target"])
    label = setup["label"]

    print("\n" + "=" * 100)
    print(f"Setup {setup_index + 1:02d}/{len(setups_df)}: {label} ({family})")
    if family == "isotropic_neuroabundance":
        print(f"Width schedule: {width_schedule(n_start, n_target).tolist()}")
    print("=" * 100)

    for repeat_idx in range(REPEATS):
        cache_path = run_cache_path(setup_id, repeat_idx)
        if CACHE_COMPLETED_RUNS and cache_path.exists() and not FORCE_RERUN:
            with open(cache_path, "rb") as f:
                cached = pkl.load(f)
            epoch_records = cached["epoch_records"]
            run_record = cached["run_record"]
            run_record["loaded_from_cache"] = True
            all_epoch_records.extend(epoch_records)
            all_run_records.append(run_record)
            print(f"repeat={repeat_idx:02d} loaded from cache: {cache_path}")
            continue

        seed = RANDOM_SEED_BASE + 10000 * setup_index + repeat_idx
        set_repeat_seed(seed)

        network = build_isotropic_network(n_start).to(DEVICE)


        optimiser = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE, weight_decay=ADAMW_WEIGHT_DECAY)


        epoch_records = []
        run_start_time = time.perf_counter()
        total_neuro_ops = 0
        total_adjustment_seconds = 0.0

        for epoch_idx in range(TOTAL_EPOCHS):
            phase = phase_for_epoch(epoch_idx)
            target_width = target_width_for_epoch(epoch_idx, n_start, n_target)
            adjustment_operations = []
            adjustment_seconds = 0.0

            if family == "isotropic_neuroabundance":
                adjustment_start = time.perf_counter()
                adjustment_operations = adjust_hidden_widths_to_target(network, optimiser, target_width, verbose=False)
                adjustment_seconds = time.perf_counter() - adjustment_start
                total_adjustment_seconds += adjustment_seconds
                total_neuro_ops += len(adjustment_operations)

            train_loss, train_accuracy = train_one_epoch(network, train_loader, optimiser, loss_fn, DEVICE, lambda_psi=PSI_DECAY)
            test_loss, test_accuracy = evaluate(network, test_loader, loss_fn, DEVICE)

            current_architecture = list(network.architecture) if hasattr(network, "architecture") else [INPUT_DIM, n_start, n_start, OUTPUT_DIM]
            actual_h1 = int(current_architecture[1])
            actual_h2 = int(current_architecture[2])

            record = {
                "setup_index": int(setup_index),
                "setup_id": setup_id,
                "family": family,
                "label": label,
                "n_start": n_start,
                "n_target": n_target,
                "repeat_idx": repeat_idx,
                "seed": seed,
                "epoch": epoch_idx + 1,
                "phase": phase,
                "target_width": int(target_width),
                "actual_h1": actual_h1,
                "actual_h2": actual_h2,
                "architecture": str(current_architecture),
                "train_loss": train_loss,
                "train_accuracy": train_accuracy,
                "test_loss": test_loss,
                "test_accuracy": test_accuracy,
                "learning_rate": float(optimiser.param_groups[0]["lr"]),
                "total_parameter_count": count_total_parameters(network),
                "nonzero_parameter_count_1e-8": count_nonzero_parameters(network, threshold=1e-8),
                "adjustment_operations": ",".join(adjustment_operations),
                "num_adjustment_operations": len(adjustment_operations),
                "adjustment_seconds": adjustment_seconds,
            }
            epoch_records.append(record)
            all_epoch_records.append(record)

            print(
                f"repeat={repeat_idx:02d} epoch={epoch_idx + 1:03d}/{TOTAL_EPOCHS} "
                f"phase={phase:<10} target={target_width:3d} h=({actual_h1:3d},{actual_h2:3d}) "
                f"test_acc={test_accuracy:6.3f}% ops={len(adjustment_operations):2d}"
            )

        run_elapsed_seconds = time.perf_counter() - run_start_time
        final_record = epoch_records[-1]
        run_record = {
            "setup_index": int(setup_index),
            "setup_id": setup_id,
            "family": family,
            "label": label,
            "n_start": n_start,
            "n_target": n_target,
            "repeat_idx": repeat_idx,
            "seed": seed,
            "final_test_accuracy": float(final_record["test_accuracy"]),
            "final_train_accuracy": float(final_record["train_accuracy"]),
            "final_test_loss": float(final_record["test_loss"]),
            "final_train_loss": float(final_record["train_loss"]),
            "final_actual_h1": int(final_record["actual_h1"]),
            "final_actual_h2": int(final_record["actual_h2"]),
            "final_total_parameter_count": int(final_record["total_parameter_count"]),
            "final_nonzero_parameter_count_1e-8": int(final_record["nonzero_parameter_count_1e-8"]),
            "best_test_accuracy": float(max(r["test_accuracy"] for r in epoch_records)),
            "best_epoch": int(max(epoch_records, key=lambda r: r["test_accuracy"])["epoch"]),
            "total_neuro_ops": int(total_neuro_ops),
            "total_adjustment_seconds": float(total_adjustment_seconds),
            "run_elapsed_seconds": float(run_elapsed_seconds),
            "loaded_from_cache": False,
            "cache_path": str(cache_path),
        }
        all_run_records.append(run_record)

        if CACHE_COMPLETED_RUNS:
            with open(cache_path, "wb") as f:
                pkl.dump({"epoch_records": epoch_records, "run_record": run_record}, f)

        print(
            f"repeat={repeat_idx:02d} complete | final_test_acc={run_record['final_test_accuracy']:.4f}% "
            f"best={run_record['best_test_accuracy']:.4f}% at epoch {run_record['best_epoch']} | "
            f"elapsed={run_elapsed_seconds/60:.2f} min"
        )

epoch_df = pd.DataFrame(all_epoch_records)
run_df = pd.DataFrame(all_run_records)

epoch_csv_path = SAVE_DIR / "Exp4_neuroabundance_epoch_history.csv"
run_csv_path = SAVE_DIR / "Exp4_neuroabundance_run_summary.csv"
epoch_df.to_csv(epoch_csv_path, index=False)
run_df.to_csv(run_csv_path, index=False)

print(f"Saved per-epoch history to: {epoch_csv_path}")
print(f"Saved per-run summary to: {run_csv_path}")
display(run_df.head())

In [ ]:
# -------------------------
# Setup-level summary tables
# -------------------------

def safe_std(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size <= 1:
        return 0.0
    return float(np.std(values, ddof=1))

summary_rows = []
for (setup_index, setup_id, family, label, n_start, n_target), group in run_df.groupby(
    ["setup_index", "setup_id", "family", "label", "n_start", "n_target"], sort=True
):
    setup_epoch_df = epoch_df[epoch_df["setup_id"].eq(setup_id)]
    epoch_mean = setup_epoch_df.groupby("epoch")["test_accuracy"].mean()
    best_mean_epoch = int(epoch_mean.idxmax())
    best_mean_accuracy = float(epoch_mean.loc[best_mean_epoch])
    summary_rows.append({
        "setup_index": int(setup_index),
        "setup_id": setup_id,
        "family": family,
        "label": label,
        "n_start": int(n_start),
        "n_target": int(n_target),
        "repeats": int(len(group)),
        "final_accuracy_mean": float(group["final_test_accuracy"].mean()),
        "final_accuracy_std": safe_std(group["final_test_accuracy"]),
        "best_repeat_accuracy_mean": float(group["best_test_accuracy"].mean()),
        "best_repeat_accuracy_std": safe_std(group["best_test_accuracy"]),
        "best_mean_epoch": best_mean_epoch,
        "best_mean_accuracy": best_mean_accuracy,
        "final_total_parameter_count_mean": float(group["final_total_parameter_count"].mean()),
        "final_total_parameter_count_std": safe_std(group["final_total_parameter_count"]),
        "final_nonzero_parameter_count_1e-8_mean": float(group["final_nonzero_parameter_count_1e-8"].mean()),
        "final_nonzero_parameter_count_1e-8_std": safe_std(group["final_nonzero_parameter_count_1e-8"]),
        "total_neuro_ops_mean": float(group["total_neuro_ops"].mean()),
        "total_neuro_ops_std": safe_std(group["total_neuro_ops"]),
        "run_elapsed_seconds_mean": float(group["run_elapsed_seconds"].mean()),
        "run_elapsed_seconds_std": safe_std(group["run_elapsed_seconds"]),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("setup_index")
summary_csv_path = SAVE_DIR / "Exp4_neuroabundance_setup_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

formatted_summary_df = summary_df.copy()
formatted_summary_df["final accuracy (%)"] = formatted_summary_df.apply(
    lambda r: f"{r['final_accuracy_mean']:.4f} ± {r['final_accuracy_std']:.4f}", axis=1
)
formatted_summary_df["best repeat accuracy (%)"] = formatted_summary_df.apply(
    lambda r: f"{r['best_repeat_accuracy_mean']:.4f} ± {r['best_repeat_accuracy_std']:.4f}", axis=1
)
formatted_summary_df["final nonzero params (>|1e-8|)"] = formatted_summary_df.apply(
    lambda r: f"{r['final_nonzero_parameter_count_1e-8_mean']:.1f} ± {r['final_nonzero_parameter_count_1e-8_std']:.1f}", axis=1
)
formatted_summary_df = formatted_summary_df[[
    "setup_index", "label", "family", "n_start", "n_target", "repeats",
    "final accuracy (%)", "best repeat accuracy (%)", "best_mean_epoch", "best_mean_accuracy",
    "final nonzero params (>|1e-8|)", "total_neuro_ops_mean",
]]
formatted_csv_path = SAVE_DIR / "Exp4_neuroabundance_formatted_summary.csv"
formatted_summary_df.to_csv(formatted_csv_path, index=False)

print(f"Saved setup-level summary to: {summary_csv_path}")
print(f"Saved formatted summary to: {formatted_csv_path}")
display(summary_df)
display(formatted_summary_df)
